# Generate qualitative text with the ValidMind library

This notebook shows how to generate qualitative documentation content directly from the ValidMind library using `vm.run_text_generation()`. Instead of switching to the UI to write text manually or trigger generation one section at a time, you can generate content for documentation text blocks programmatically from within a notebook and log it back to the corresponding sections of the model document.

After building an example model and documenting its quantitative results, we’ll show how to generate text for individual content blocks, customize the output with prompts, control the context used for generation, and apply the same pattern across the full template. By the end, you’ll have an end-to-end example of how quantitative test results and AI-generated qualitative content can work together to populate a full model document from Python, giving you a more automated documentation workflow directly in the library.

::: {.content-hidden when-format="html"}
## Contents    
- [About ValidMind](#toc1__)    
  - [Before you begin](#toc1_1__)    
  - [New to ValidMind?](#toc1_2__)    
  - [Key concepts](#toc1_3__)    
- [Setting up](#toc2__)    
  - [Install the ValidMind Library](#toc2_1__)    
  - [Initialize the ValidMind Library](#toc2_2__)    
    - [Register sample model](#toc2_2_1__)    
    - [Apply documentation template](#toc2_2_2__)    
    - [Get your code snippet](#toc2_2_3__)    
  - [Initialize the Python environment](#toc2_3__)    
- [Getting to know ValidMind](#toc3__)    
  - [Preview the documentation template](#toc3_1__)    
  - [View model documentation in the ValidMind Platform](#toc3_2__)    
- [Build the example model](#toc4__)    
  - [Import the sample dataset](#toc4_1__)    
  - [Preprocessing the raw dataset](#toc4_2__)    
  - [Training an XGBoost classifier model](#toc4_3__)    
- [Initialize the ValidMind inputs](#toc5__)    
- [Document test results](#toc6__)    
- [Document qualitative sections](#toc7__)    
  - [Generate text for a single content block](#toc7_1__)    
  - [Customize the prompt](#toc7_2__)    
  - [Pass section-specific context](#toc7_3__)    
  - [Populate all text blocks](#toc7_4__)    
- [In summary](#toc8__)    
- [Next steps](#toc9__)    
  - [Work with your model documentation](#toc9_1__)    
  - [Discover more learning resources](#toc9_2__)    
- [Upgrade ValidMind](#toc10__)    

:::
<!-- jn-toc-notebook-config
	numbering=false
	anchor=true
	flat=false
	minLevel=2
	maxLevel=4
	/jn-toc-notebook-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

<a id='toc1__'></a>

## About ValidMind

ValidMind is a suite of tools for managing model risk, including risk associated with AI and statistical models. 

You use the ValidMind Library to automate documentation and validation tests, and then use the ValidMind Platform to collaborate on model documentation. Together, these products simplify model risk management, facilitate compliance with regulations and institutional standards, and enhance collaboration between yourself and model validators.

<a id='toc1_1__'></a>

### Before you begin

This notebook assumes you have basic familiarity with Python, including an understanding of how functions work. If you are new to Python, you can still run the notebook but we recommend further familiarizing yourself with the language. 

If you encounter errors due to missing modules in your Python environment, install the modules with `pip install`, and then re-run the notebook. For more help, refer to [Installing Python Modules](https://docs.python.org/3/installing/index.html).

<a id='toc1_2__'></a>

### New to ValidMind?

If you haven't already seen our documentation on the [ValidMind Library](https://docs.validmind.ai/developer/validmind-library.html), we recommend you begin by exploring the available resources in this section. There, you can learn more about documenting models and running tests, as well as find code samples and our Python Library API reference.

<div class="alert alert-block alert-info" style="background-color: #B5B5B510; color: black; border: 1px solid #083E44; border-left-width: 5px; box-shadow: 2px 2px 4px rgba(0, 0, 0, 0.2);border-radius: 5px;"><span style="color: #083E44;"><b>For access to all features available in this notebook, you'll need access to a ValidMind account.</b></span>
<br></br>
<a href="https://docs.validmind.ai/guide/configuration/register-with-validmind.html" style="color: #DE257E;"><b>Register with ValidMind</b></a></div>

<a id='toc1_3__'></a>

### Key concepts

**Validation report**: A comprehensive and structured assessment of a model’s development and performance, focusing on verifying its integrity, appropriateness, and alignment with its intended use. It includes analyses of model assumptions, data quality, performance metrics, outcomes of testing procedures, and risk considerations. The validation report supports transparency, regulatory compliance, and informed decision-making by documenting the validator’s independent review and conclusions.

**Validation report template**: Serves as a standardized framework for conducting and documenting model validation activities. It outlines the required sections, recommended analyses, and expected validation tests, ensuring consistency and completeness across validation reports. The template helps guide validators through a systematic review process while promoting comparability and traceability of validation outcomes.

**Tests**: A function contained in the ValidMind Library, designed to run a specific quantitative test on the dataset or model. Tests are the building blocks of ValidMind, used to evaluate and document models and datasets.

**Metrics**: A subset of tests that do not have thresholds. In the context of this notebook, metrics and tests can be thought of as interchangeable concepts.

**Custom metrics**: Custom metrics are functions that you define to evaluate your model or dataset. These functions can be registered with the ValidMind Library to be used in the ValidMind Platform.

**Inputs**: Objects to be evaluated and documented in the ValidMind Library. They can be any of the following:

  - **model**: A single model that has been initialized in ValidMind with [`vm.init_model()`](https://docs.validmind.ai/validmind/validmind.html#init_model).
  - **dataset**: Single dataset that has been initialized in ValidMind with [`vm.init_dataset()`](https://docs.validmind.ai/validmind/validmind.html#init_dataset).
  - **models**: A list of ValidMind models - usually this is used when you want to compare multiple models in your custom metric.
  - **datasets**: A list of ValidMind datasets - usually this is used when you want to compare multiple datasets in your custom metric. (Learn more: [Run tests with multiple datasets](https://docs.validmind.ai/notebooks/how_to/tests/run_tests/configure_tests/run_tests_that_require_multiple_datasets.html))

**Parameters**: Additional arguments that can be passed when running a ValidMind test, used to pass additional information to a metric, customize its behavior, or provide additional context.

**Outputs**: Custom metrics can return elements like tables or plots. Tables may be a list of dictionaries (each representing a row) or a pandas DataFrame. Plots may be matplotlib or plotly figures.

<a id='toc2__'></a>

## Setting up

<a id='toc2_1__'></a>

### Install the ValidMind Library

<div class="alert alert-block alert-info" style="background-color: #B5B5B510; color: black; border: 1px solid #083E44; border-left-width: 5px; box-shadow: 2px 2px 4px rgba(0, 0, 0, 0.2);border-radius: 5px;"><span style="color: #083E44;"><b>Recommended Python versions</b></span>
<br></br>
Python 3.8 <= x <= 3.14</div>

To install the library:

In [ ]:
%pip install -q validmind

<a id='toc2_2__'></a>

### Initialize the ValidMind Library

<a id='toc2_2_1__'></a>

#### Register sample model

Let's first register a sample model for use with this notebook:

1. In a browser, [log in to ValidMind](https://docs.validmind.ai/guide/configuration/log-in-to-validmind.html).

2. In the left sidebar, navigate to **Inventory** and click **+ Register Model**.

3. Enter the model details and click **Next >** to continue to assignment of model stakeholders. ([Need more help?](https://docs.validmind.ai/guide/model-inventory/register-models-in-inventory.html))

4. Select your own name under the **MODEL OWNER** drop-down.

5. Click **Register Model** to add the model to your inventory.

<a id='toc2_2_2__'></a>

#### Apply documentation template

Once you've registered your model, let's select a documentation template. A template predefines sections for your model documentation and provides a general outline to follow, making the documentation process much easier.

1. In the left sidebar that appears for your model, click **Documents** and select **Development**.

2. Under **TEMPLATE**, select `Customer Churn`.

3. Click **Use Template** to apply the template.

<a id='toc2_2_3__'></a>

#### Get your code snippet

Initialize the ValidMind Library with the *code snippet* unique to each model per document, ensuring your test results are uploaded to the correct model and automatically populated in the right document in the ValidMind Platform when you run this notebook.

1. On the left sidebar that appears for your model, select **Getting Started** and select `Development` from the **DOCUMENT** drop-down menu.
2. Click **Copy snippet to clipboard**.
3. Next, [load your model identifier credentials from an `.env` file](https://docs.validmind.ai/developer/model-documentation/store-credentials-in-env-file.html) or replace the placeholder with your own code snippet:

In [ ]:
# Load your model identifier credentials from an `.env` file

%load_ext dotenv
%dotenv .env

# Or replace with your code snippet

import validmind as vm

vm.init(
    api_host="http://localhost:5000/api/v1/tracking",
    api_key="...",
    api_secret="...",
    document="documentation", # requires library >=2.12.0
    model="...",
)

<a id='toc2_3__'></a>

### Initialize the Python environment

Then, let's import the necessary libraries and set up your Python environment for data analysis:

- Import **Extreme Gradient Boosting** (XGBoost) with an alias so that we can reference its functions in later calls. XGBoost is a powerful machine learning library designed for speed and performance, especially in handling structured or tabular data.
- Enable **`matplotlib`**, a plotting library used for visualizing data. Ensures that any plots you generate will render inline in our notebook output rather than opening in a separate window.

In [ ]:
%matplotlib inline

import xgboost as xgb

<a id='toc3__'></a>

## Getting to know ValidMind

<a id='toc3_1__'></a>

### Preview the documentation template

Let's verify that you have connected the ValidMind Library to the ValidMind Platform and that the appropriate *template* is selected for your model.

You will upload documentation and test results unique to your model based on this template later on. For now, **take a look at the default structure that the template provides with [the `vm.preview_template()` function](https://docs.validmind.ai/validmind/validmind.html#preview_template)** from the ValidMind library and note the empty sections:

In [ ]:
vm.preview_template()

<a id='toc3_2__'></a>

### View model documentation in the ValidMind Platform

Next, let's head to the ValidMind Platform to see the template in action:

1. In a browser, [log in to ValidMind](https://docs.validmind.ai/guide/configuration/log-in-to-validmind.html).

2. In the left sidebar, navigate to **Inventory** and select the model you registered for this notebook.

3. Click **Development** under Documents for your model and note how the structure of the documentation matches our preview above.

<a id='toc4__'></a>

## Build the example model

<a id='toc4_1__'></a>

### Import the sample dataset

First, let's import the public [Bank Customer Churn Prediction](https://www.kaggle.com/datasets/shantanudhakadd/bank-customer-churn-prediction) dataset from Kaggle so that we have something to work with.

In our below example, note that: 

- The target column, `Exited` has a value of `1` when a customer has churned and `0` otherwise.
- The ValidMind Library provides a wrapper to automatically load the dataset as a [Pandas DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) object. A Pandas Dataframe is a two-dimensional tabular data structure that makes use of rows and columns.

In [ ]:
from validmind.datasets.classification import customer_churn

print(
    f"Loaded demo dataset with: \n\n\t• Target column: '{customer_churn.target_column}' \n\t• Class labels: {customer_churn.class_labels}"
)

raw_df = customer_churn.load_data()
raw_df.head()

<a id='toc4_2__'></a>

### Preprocessing the raw dataset

In this section, we preprocess the raw dataset so it is ready for model training and validation. This includes splitting the data into training, validation, and test subsets to support both model fitting and evaluation on unseen data, and then separating each subset into input features and target labels so the model can learn from customer attributes and predict whether a customer churned.

In [ ]:
train_df, validation_df, test_df = customer_churn.preprocess(raw_df)

x_train = train_df.drop(customer_churn.target_column, axis=1)
y_train = train_df[customer_churn.target_column]
x_val = validation_df.drop(customer_churn.target_column, axis=1)
y_val = validation_df[customer_churn.target_column]

<a id='toc4_3__'></a>

### Training an XGBoost classifier model

In this section, we train an XGBoost classifier to predict customer churn, using early stopping to halt training if performance does not improve after 10 rounds and reduce unnecessary fitting. We configure the model to evaluate performance with three complementary metrics: error for incorrect predictions, logloss for prediction confidence, and auc for class separation. The model is trained on the training split and evaluated against the validation split during fitting, while verbose=False keeps the training output concise.

In [ ]:
model = xgb.XGBClassifier(early_stopping_rounds=10)

model.set_params(
    eval_metric=["error", "logloss", "auc"],
)

model.fit(
    x_train,
    y_train,
    eval_set=[(x_val, y_val)],
    verbose=False,
)

<a id='toc5__'></a>

## Initialize the ValidMind inputs

We begin by registering the datasets and trained model as ValidMind inputs so they can be referenced consistently throughout the documentation workflow. For the datasets, this means creating ValidMind Dataset objects for the raw, training, and testing data, each with a unique `input_id` for traceability. Where needed, we also provide supporting metadata such as the target column and class labels so tests can interpret the data correctly.

In [ ]:
# Initialize the raw dataset
vm_raw_dataset = vm.init_dataset(
    dataset=raw_df,
    input_id="raw_dataset",
    target_column=customer_churn.target_column,
    class_labels=customer_churn.class_labels,
)

# Initialize the training dataset
vm_train_ds = vm.init_dataset(
    dataset=train_df,
    input_id="train_dataset",
    target_column=customer_churn.target_column,
)

# Initialize the testing dataset
vm_test_ds = vm.init_dataset(
    dataset=test_df,
    input_id="test_dataset",
    target_column=customer_churn.target_column
)

Next, we initialize a ValidMind model object with `vm.init_model()`. This creates a standardized representation of the trained model that can be passed into ValidMind tests and other library functions, making it possible to evaluate the model and connect its results to the documentation.

In [ ]:
# Initialize the model
vm_model = vm.init_model(
    model,
    input_id="model",
)

Finally, we assign predictions from the trained model to the training and testing datasets. The `assign_predictions()` method links predicted classes and probabilities to each dataset, and can also compute predictions automatically if they are not passed explicitly. This step is what allows ValidMind to run performance and diagnostic tests using the model outputs.

In [ ]:
vm_train_ds.assign_predictions(
    model=vm_model,
)
vm_test_ds.assign_predictions(
    model=vm_model,
)

<a id='toc6__'></a>

## Document test results

In this section, we run the documentation tests defined by the applied template to populate the quantitative parts of the model documentation. The `vm.run_documentation_tests()` function discovers each test-driven block in the template, executes the corresponding tests, and uploads the resulting artifacts to the ValidMind Platform.

To run the full suite successfully, ValidMind needs to know which model and dataset inputs should be used for each test. This can be done with a shared `inputs` argument when all tests use the same objects, or with a `config` dictionary when individual tests require specific inputs or parameters. In this example, we use the default test parameters and provide the input configuration needed for the demo model.

In [ ]:
from validmind.utils import preview_test_config

test_config = customer_churn.get_demo_test_config()
preview_test_config(test_config)

Once the configuration is prepared, we pass it to `vm.run_documentation_tests()` and execute the full suite. The returned `full_suite` object contains the test results and represents the quantitative documentation that has been generated for the model.

In [ ]:
full_suite = vm.run_documentation_tests(config=test_config)

<a id='toc7__'></a>

## Document qualitative sections

In addition to documenting quantitative results through tests, ValidMind now supports programmatic generation of qualitative content for the text blocks in a model documentation template through `vm.run_text_generation()`. This function allows you to generate AI-assisted text for a specific content block directly from a notebook and then log it back to the corresponding section of the document. As a result, you can populate qualitative sections without switching to the UI to write text manually or trigger generation one section at a time.

In the next sections, we’ll walk through the main ways to use this functionality. We’ll start by generating text for a single content block with the default behavior, then show how to customize the output with a prompt, how to control the context used for generation by selecting specific sections, and finally how to scale the same pattern across all text blocks in the document.

<a id='toc7_1__'></a>

### Generate text for a single content block

First, we’ll use `vm.run_text_generation()` to generate qualitative text for a single documentation block. By providing a `content_id`, you can target the exact text placeholder you want to populate and let ValidMind generate content using the current document context. The helper `vm.get_content_ids()` is useful for inspecting which content blocks are available in the active template, making it easier to identify the IDs you can use when generating and logging text programmatically.

In [ ]:
vm.get_content_ids()

In [ ]:
vm.run_text_generation(
    content_id="dataset_summary_text",
).log()

<a id='toc7_2__'></a>

### Customize the prompt

Next, we’ll customize the generated output by passing a `prompt` to `vm.run_text_generation()`. This makes it possible to guide not just the subject of the generated text, but also its structure, tone, level of detail, and presentation format. In practice, this allows you to tailor the output for different documentation needs, such as producing a short narrative summary, a more structured section, or content written for a specific audience, while still relying on the same underlying document context for generation.

In [ ]:
prompt = """
Use exactly this structure:

<h3>Dataset Overview</h3>
<p>Explain in 1-2 sentences what the dataset contains and what it is used for.</p>

<h3>Dataset Summary</h3>
<p>Summarize the dataset structure, target outcome, and the main types of input features in 2-3 sentences.</p>

<h3>Key Characteristics</h3>
<ul>
  <li>Include 2-3 concise points about the most important characteristics of the dataset.</li>
</ul>

<h3>Data Quality and Considerations</h3>
<ul>
  <li>Include 2-3 concise points about important quality observations, limitations, or considerations relevant to the dataset.</li>
</ul>

<h3>Overall Assessment</h3>
<p>End with a short balanced conclusion on the dataset's suitability for model development and evaluation.</p>
"""

In [ ]:
vm.run_text_generation(
    content_id="dataset_summary_text",
    prompt=prompt,
).log()

<a id='toc7_3__'></a>

### Pass section-specific context

Then, we’ll control the `context` used for generation by passing a selected set of content IDs to `vm.run_text_generation()`. Rather than relying on the full document, this lets you focus the model on the most relevant parts of the documentation for a given text block. In practice, that means you can generate more targeted qualitative content by choosing which existing test and text blocks should inform the output.

In [ ]:
vm.get_content_ids("data_description")

In [ ]:
vm.run_text_generation(
    content_id="dataset_summary_text",
    context={"content_ids": vm.get_content_ids("data_description")},
).log()

### Append a new text block to a section

Sometimes you may want to generate text for a `content_id` that is not already defined in the template. In that case, you can still generate the text with `vm.run_text_generation()` and then use `.log(section_id=...)` to tell ValidMind where that new text block should be placed in the document. 

In [ ]:
vm.run_text_generation(
    content_id="monitoring_implementation",
    #section_id="regulatory_requirements",
    prompt="Write two single paragraphs.",
).log()

<a id='toc7_4__'></a>

### Populate all text blocks

Finally, we’ll scale this pattern across the full document by generating text for all documentation text blocks in a loop. A simple helper maps each target content_id to the most relevant section context, resolves those section IDs into content IDs, and applies `vm.run_text_generation()` consistently across the template. This makes it possible to programmatically populate all qualitative sections of the document from the notebook in a single workflow.

The helper below runs the same generation pattern across multiple text blocks. `_build_context()` takes a list of section IDs and resolves them into the `content_ids` format expected by `vm.run_text_generation()`, while `generate_documentation_text()` loops through a configuration dictionary, runs generation for each target `content_id`, and logs the result. The `config` dictionary maps each text block we want to populate and the section IDs we want to use as context for that block. When a value is set to `None`, the generation falls back to using the full document as context.

In [ ]:
def _build_context(section_ids):
    return {"content_ids": vm.get_content_ids(section_ids=section_ids)}


def generate_documentation_text(config, show=True):
    results = {}

    for content_id, section_ids in config.items():
        result = vm.run_text_generation(
            content_id=content_id,
            prompt="Write a single paragraph with no headings, bullets, or tables.",
            context=_build_context(section_ids),
            show=show,
        )
        result.log()
        results[content_id] = result

    return results


config = {
    "dataset_summary_text": ["data_description", "descriptive_statistics"],
    "data_quality_tests_text": ["data_description"],
    "feature_selection": [
        "data_description",
        "descriptive_statistics",
        "correlations",
        "feature_selection",
    ],
    "model_validation_tests_text": ["model_training", "model_evaluation"],
    "validmind.model_validation.sklearn.SHAPGlobalImportance_global_importance_text": [
        "feature_selection",
        "model_evaluation",
        "explainability",
    ],
    "model_weak_spots_description": ["model_evaluation", "model_diagnosis"],
    "model_overfit_regions_description": [
        "model_training",
        "model_evaluation",
        "model_diagnosis",
    ],
    "model_robustness_description": ["model_evaluation", "model_diagnosis"],
    "monitoring_plan": None,
    "monitoring_implementation": None,
    "governance_plan": None,
}

In [ ]:
generate_documentation_text(config)

<a id='toc8__'></a>

## In summary

In this notebook, you learned how to:

- [x] Build and document an example customer churn model with ValidMind
- [x] Run documentation tests to populate the quantitative sections of a model document
- [x] Generate qualitative text for a single documentation content block with `vm.run_text_generation()`
- [x] Customize generated output by passing a prompt
- [x] Control generation context by selecting specific sections of the document
- [x] Populate all documentation text blocks programmatically from a notebook

<a id='toc9__'></a>

## Next steps

You can look at the output produced by the ValidMind Library right in the notebook where you ran the code, as you would expect. But there is a better way — use the ValidMind Platform to work with your model documentation.

<a id='toc9_1__'></a>

### Work with your model documentation

1. From the **Inventory** in the ValidMind Platform, go to the model you registered earlier. ([Need more help?](https://docs.validmind.ai/guide/model-inventory/working-with-model-inventory.html))

2. In the left sidebar that appears for your model, click **Development** under Documents.

What you see is the full draft of your model documentation in a more easily consumable version. From here, you can make qualitative edits to model documentation, view guidelines, collaborate with validators, and submit your model documentation for approval when it's ready. [Learn more ...](https://docs.validmind.ai/guide/working-with-model-documentation.html)

<a id='toc9_2__'></a>

### Discover more learning resources

For a more in-depth introduction to using the ValidMind Library for development, check out our introductory development series and the accompanying interactive training:

- **[ValidMind for model development](https://docs.validmind.ai/developer/validmind-library.html#for-model-development)**
- **[Developer Fundamentals](https://docs.validmind.ai/training/developer-fundamentals/developer-fundamentals-register.html)**

We also offer many interactive notebooks to help you document models:

- [Run tests & test suites](https://docs.validmind.ai/developer/how-to/testing-overview.html)
- [Use ValidMind Library features](https://docs.validmind.ai/developer/how-to/feature-overview.html)
- [Code samples by use case](https://docs.validmind.ai/guide/samples-jupyter-notebooks.html)

Or, visit our [documentation](https://docs.validmind.ai/) to learn more about ValidMind.

<a id='toc10__'></a>

## Upgrade ValidMind

<div class="alert alert-block alert-info" style="background-color: #B5B5B510; color: black; border: 1px solid #083E44; border-left-width: 5px; box-shadow: 2px 2px 4px rgba(0, 0, 0, 0.2);border-radius: 5px;">After installing ValidMind, you’ll want to periodically make sure you are on the latest version to access any new features and other enhancements.</div>

Retrieve the information for the currently installed version of ValidMind:

In [ ]:
%pip show validmind

If the version returned is lower than the version indicated in our [production open-source code](https://github.com/validmind/validmind-library/blob/prod/validmind/__version__.py), restart your notebook and run:

```bash
%pip install --upgrade validmind
```

You may need to restart your kernel after running the upgrade package for changes to be applied.

<!-- VALIDMIND COPYRIGHT -->

<small>

***

Copyright © 2023-2026 ValidMind Inc. All rights reserved.<br>
Refer to [LICENSE](https://github.com/validmind/validmind-library/blob/main/LICENSE) for details.<br>
SPDX-License-Identifier: AGPL-3.0 AND ValidMind Commercial</small>